# T2.3 – Mapping Units of Measurement

## Vienna Weather Wet-Month Prediction Experiment

This notebook assigns ontology-based unit mappings to all numeric attributes
in the DBRepo schema.

The mappings support the FAIR principles by improving:
- semantic interoperability,
- machine readability,
- metadata quality,
- and reusability of the experiment data.

## Ontology Choice

The recommended ontology for this task was the SI Digital Framework.
For practical integration with the DBRepo test instance, the OM-2 ontology
(Ontology of Units of Measure) was selected because it is already registered
and supported within the DBRepo metadata registry.

OM-2 concepts are used for:
- temperature
- atmospheric pressure
- precipitation
- wind speed
- humidity
- temporal units
- spatial coordinates
- dimensionless identifiers
- count-based quantities

## Dataset

Source dataset:
Stadt Wien – Monthly weather observations at Hohe Warte station since 1872.

License:
CC BY 4.0

## 1. Import libraries and configure DBRepo connection

We connect to the DBRepo instance using the Python REST client.
The database and table identifiers were created previously in T2.1.

In [43]:
import sys
import importlib.metadata
import dbrepo

print("Python:", sys.executable)
print("dbrepo version:", importlib.metadata.version("dbrepo"))
print("dbrepo module path:", dbrepo.__file__)

Python: /opt/anaconda3/bin/python
dbrepo version: 1.13.8
dbrepo module path: /opt/anaconda3/lib/python3.13/site-packages/dbrepo/__init__.py


In [44]:
import pandas as pd
import requests
from dbrepo.RestClient import RestClient

In [45]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at"

USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"

DATABASE_ID = "a181cad5-4bdb-48b2-937e-3e75293f6a7b"

TABLE_IDS = {
    "weather_measurement_v2": "3674fea3-a7be-4dfe-8356-bc692bd1ff6c",
    "time_dimension": "fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde",
    "station": "ab02386c-e27c-4c1f-a27d-93034ce3fa79"
}

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Connected to DBRepo.")

Connected to DBRepo.


## 2. Load the unit mapping table

The ontology mappings are maintained in a CSV file to ensure:
- transparency,
- reproducibility,
- and easier maintenance of semantic metadata.

Each row contains:
- table name,
- column name,
- ontology URI,
- human-readable unit label.

In [46]:
mapping_df = pd.read_csv("../docs/unit_mapping.csv")

mapping_df.head()

,table_name,column_name,unit_uri,unit_label
0,weather_measurement_v2,measurement_id,http://www.ontology-of-units-of-measure.org/re...,unitless
1,weather_measurement_v2,station_num,http://www.ontology-of-units-of-measure.org/re...,unitless
2,weather_measurement_v2,time_id,http://www.ontology-of-units-of-measure.org/re...,unitless
3,weather_measurement_v2,t_mean_c,http://www.ontology-of-units-of-measure.org/re...,degree Celsius
4,weather_measurement_v2,t_max_c,http://www.ontology-of-units-of-measure.org/re...,degree Celsius


## 3. Validate DB schema

Before assigning ontology mappings, we verify that all referenced
tables and columns exist in the DBRepo schema.

In [47]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    print(t.name, t.id)

weather_measurement_v2 3674fea3-a7be-4dfe-8356-bc692bd1ff6c
weather_measurement 631c878e-1f39-47a0-be38-0b3c0e2733c8
time_dimension fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde
station ab02386c-e27c-4c1f-a27d-93034ce3fa79


## 4. Validate unit mappings

This step validates that:
- all mapped tables exist,
- all mapped columns exist,
- and every numeric attribute has a corresponding ontology-based unit mapping.

In [48]:
success = 0
failed = 0

for _, row in mapping_df.iterrows():

    table_name = row["table_name"]
    column_name = row["column_name"]
    unit_uri = row["unit_uri"]

    table_id = TABLE_IDS[table_name]

    try:

        table = client.get_table(DATABASE_ID, table_id)

        col = next(
            (c for c in table.columns if c.name == column_name),
            None
        )

        if col is None:
            print(f"FAILED: {table_name}.{column_name} not found")
            failed += 1

        else:
            print(
                f"OK: {table_name}.{column_name} "
                f"→ {unit_uri}"
            )
            success += 1

    except Exception as e:
        print(f"FAILED ({e}): {table_name}.{column_name}")
        failed += 1

print("\n===================================")
print(f"Validated mappings: {success}")
print(f"Failed mappings   : {failed}")
print("===================================")

OK: weather_measurement_v2.measurement_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.station_num → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.time_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.t_mean_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.mean_t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.mean_t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.p_mean_hpa → http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal
OK: weather_measurement_v2

## 5. Attempt DBRepo metadata integration

As required by the assignment, we attempt to assign ontology-based
unit mappings directly to DBRepo columns using the REST API.

In [51]:
import requests

def get_table_json(database_id, table_id):
    url = f"{ENDPOINT}/api/v1/database/{database_id}/table/{table_id}"
    response = requests.get(
        url,
        auth=(USERNAME, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    return response.json()


def get_column_json(table_json, column_name):
    for column in table_json.get("columns", []):
        if column.get("name") == column_name:
            return column
    return None

In [54]:
success = 0
failed = 0

for _, row in mapping_df.iterrows():
    table_name = row["table_name"]
    column_name = row["column_name"]
    unit_uri = row["unit_uri"]

    table_id = TABLE_IDS[table_name]

    try:
        table = client.get_table(DATABASE_ID, table_id)
        col = next((c for c in table.columns if c.name == column_name), None)

        if col is None:
            print(f"FAILED (not found): {table_name}.{column_name}")
            failed += 1
            continue

        # Fetch current REST metadata to preserve existing concept_uri
        table_json = get_table_json(DATABASE_ID, table_id)
        column_json = get_column_json(table_json, column_name)

        if column_json is None:
            print(f"FAILED (REST metadata not found): {table_name}.{column_name}")
            failed += 1
            continue

        existing_concept_uri = column_json.get("concept_uri")

        # IMPORTANT:
        # Pass both concept_uri and unit_uri so unit update does not erase concept mapping.
        updated_col = client.update_table_column(
            database_id=DATABASE_ID,
            table_id=table_id,
            column_id=col.id,
            concept_uri=existing_concept_uri,
            unit_uri=unit_uri
        )

        print(f"OK: {table_name}.{column_name}")
        print(f"  concept_uri preserved: {existing_concept_uri}")
        print(f"  unit_uri updated     : {unit_uri}")

        success += 1

    except Exception as e:
        print(f"FAILED ({type(e).__name__}: {e}): {table_name}.{column_name}")
        failed += 1

print(f"\nUpload success : {success}")
print(f"Upload failed  : {failed}")

OK: weather_measurement_v2.measurement_id
  concept_uri preserved: http://purl.org/dc/terms/identifier
  unit_uri updated     : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.station_num
  concept_uri preserved: http://purl.org/dc/terms/identifier
  unit_uri updated     : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.time_id
  concept_uri preserved: http://purl.org/dc/terms/identifier
  unit_uri updated     : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.t_mean_c
  concept_uri preserved: http://qudt.org/vocab/quantitykind/Temperature
  unit_uri updated     : http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_max_c
  concept_uri preserved: http://qudt.org/vocab/quantitykind/Temperature
  unit_uri updated     : http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_mi

In [55]:
import requests

TABLE_IDS_TO_CHECK = {
    "station": TABLE_IDS["station"],
    "time_dimension": TABLE_IDS["time_dimension"],
    "weather_measurement_v2": TABLE_IDS["weather_measurement_v2"],
}

def get_table_json(database_id, table_id):
    url = f"{ENDPOINT}/api/v1/database/{database_id}/table/{table_id}"
    response = requests.get(
        url,
        auth=(USERNAME, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    return response.json()


for table_name, table_id in TABLE_IDS_TO_CHECK.items():
    table_json = get_table_json(DATABASE_ID, table_id)

    print("\n" + "=" * 80)
    print(f"Table: {table_name}")
    print(f"Table ID: {table_id}")
    print("=" * 80)

    for column in table_json.get("columns", []):
        column_name = column.get("name")
        unit_uri = column.get("unit_uri")
        print(f"{column_name}: {unit_uri}")


Table: station
Table ID: ab02386c-e27c-4c1f-a27d-93034ce3fa79
station_num: http://www.ontology-of-units-of-measure.org/resource/om-2/one
nuts_code: None
district_code: http://www.ontology-of-units-of-measure.org/resource/om-2/one
sub_district_code: http://www.ontology-of-units-of-measure.org/resource/om-2/one
station_name: None
latitude_deg: http://www.ontology-of-units-of-measure.org/resource/om-2/degree
longitude_deg: http://www.ontology-of-units-of-measure.org/resource/om-2/degree
altitude_m: http://www.ontology-of-units-of-measure.org/resource/om-2/metre

Table: time_dimension
Table ID: fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde
time_id: http://www.ontology-of-units-of-measure.org/resource/om-2/one
ref_year: http://www.ontology-of-units-of-measure.org/resource/om-2/year
ref_month: http://www.ontology-of-units-of-measure.org/resource/om-2/month

Table: weather_measurement_v2
Table ID: 3674fea3-a7be-4dfe-8356-bc692bd1ff6c
measurement_id: http://www.ontology-of-units-of-measure.org/resourc